# 01 — Chest X-ray Classification : carnet d'experimentation pre-production

**Auteur :** Yann Smatti  
**Date de travail :** Mars 2026  
**Dataset cible :** Chest X-Ray Images (Pneumonia)  
**Objectif :** evaluer une baseline de classification binaire exploitable comme point de comparaison avant industrialisation

## Question de travail

Je cherche ici a etablir une baseline defendable pour un cas d'usage de screening `NORMAL` vs `PNEUMONIA`, avant toute integration API ou interface clinique. Le but n'est pas de montrer un score flatteur, mais de verifier si le signal porte effectivement sur la pathologie et si le pipeline de donnees est suffisamment propre pour justifier une phase d'entrainement suivie et tracee.

## Hypotheses a verifier

- le split fourni ne contient pas d'anomalie evidente de distribution ou de fuite de donnees ;
- un backbone ImageNet simple permet d'obtenir une baseline robuste sans ingenierie excessive ;
- la sensibilite sur `PNEUMONIA` est plus importante que l'accuracy brute pour la suite du projet ;
- si les erreurs se concentrent sur des cas ambigus ou de faible qualite, une phase d'amelioration par calibration et revue d'erreurs est justifiee.

## Ce que ce notebook doit produire

1. un controle qualite des donnees et des tailles d'images ;
2. une baseline entrainable et reproductible ;
3. une lecture exploitable des courbes, de la ROC et de la matrice de confusion ;
4. une decision claire : continuer, corriger les donnees ou abandonner cette formulation telle quelle.

In [ ]:
## 1. Configuration & imports
import warnings; warnings.filterwarnings('ignore')

from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from PIL import Image

import tensorflow as tf
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    ConfusionMatrixDisplay
)

from src.utils.config import load_config
from src.preprocessing.image_loader import load_and_preprocess_image

# ── Config ──────────────────────────────────────────────────────────────────
cfg       = load_config('configs/config.yaml')
DATA_DIR  = Path(cfg['dataset_dir'])
IMG_SIZE  = cfg.get('image_size', 224)
BATCH     = cfg.get('batch_size', 32)
CLASSES   = ['NORMAL', 'PNEUMONIA']

print("Config chargée :")
for k, v in cfg.items():
    print(f"  {k:<25} {v}")
print(f"\nDataset dir : {DATA_DIR}  |  existe = {DATA_DIR.exists()}")
print(f"TensorFlow  : {tf.__version__}")


In [ ]:
## 2. Inspection de l'arborescence
all_files = sorted(DATA_DIR.rglob('*')) if DATA_DIR.exists() else []
image_files = [p for p in all_files if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}]

print(f"Fichiers totaux    : {len(all_files)}")
print(f"Images trouvées    : {len(image_files)}")
print(f"Types de fichiers  : {dict(Counter(p.suffix.lower() for p in all_files))}")
print()

# Arborescence splits × classes
splits_stats = {}
for split in ['train', 'val', 'test', 'Train', 'Val', 'Test']:
    split_dir = DATA_DIR / split
    if split_dir.exists():
        splits_stats[split.lower()] = {
            cls.name: sum(1 for p in cls.glob('*') if p.is_file())
            for cls in sorted(split_dir.iterdir()) if cls.is_dir()
        }

if splits_stats:
    df_splits = pd.DataFrame(splits_stats).T.fillna(0).astype(int)
    print("Distribution par split et classe :")
    print(df_splits)
    print()
    # Ratio NORMAL/PNEUMONIA
    for sp, counts in splits_stats.items():
        total = sum(counts.values())
        ratio = counts.get('PNEUMONIA', 0) / max(counts.get('NORMAL', 1), 1)
        print(f"  {sp:6s}  total={total:<6}  PNEUM/NORMAL={ratio:.2f}")
else:
    print("⚠ Dataset absent — télécharge avec scripts/download_dataset.ps1")


In [ ]:
## 3. Distribution des classes — visualisation
if splits_stats:
    fig, axes = plt.subplots(1, len(splits_stats), figsize=(4 * len(splits_stats), 4))
    if len(splits_stats) == 1:
        axes = [axes]
    colors = {'NORMAL': '#3498db', 'PNEUMONIA': '#e74c3c'}

    for ax, (split_name, counts) in zip(axes, splits_stats.items()):
        names  = list(counts.keys())
        values = list(counts.values())
        bars = ax.bar(names, values, color=[colors.get(n, '#95a5a6') for n in names])
        ax.set_title(f'Split : {split_name}', fontweight='bold')
        ax.set_ylabel('Nombre d\'images')
        for bar, val in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                    str(val), ha='center', va='bottom', fontsize=10)

    plt.suptitle('Distribution des classes — Chest X-ray Dataset', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Dataset absent — visualisation impossible.")


In [ ]:
## 4. Visualisation de la grille d'images brutes
if image_files:
    # Séparer par classe
    normal_imgs = [p for p in image_files if 'NORMAL' in str(p).upper()]
    pneumo_imgs = [p for p in image_files if 'PNEUMONIA' in str(p).upper()]

    np.random.seed(42)
    sample_n = np.random.choice(normal_imgs, min(6, len(normal_imgs)), replace=False)
    sample_p = np.random.choice(pneumo_imgs, min(6, len(pneumo_imgs)), replace=False)
    samples  = list(zip(['NORMAL'] * 6, sample_n)) + list(zip(['PNEUMONIA'] * 6, sample_p))

    fig, axes = plt.subplots(2, 6, figsize=(16, 6))
    for ax, (label, path) in zip(axes.ravel(), samples):
        img = Image.open(path).convert('L')  # niveaux de gris (radiologie)
        ax.imshow(img, cmap='gray')
        color = '#3498db' if label == 'NORMAL' else '#e74c3c'
        ax.set_title(label, color=color, fontsize=8, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Échantillons — Chest X-ray (rangée 1 : NORMAL, rangée 2 : PNEUMONIA)',
                 fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"NORMAL samples : {len(normal_imgs)} images")
    print(f"PNEUMONIA samples : {len(pneumo_imgs)} images")
else:
    print("Aucune image trouvée.")


In [ ]:
## 5. Vérification du pipeline de prétraitement
if image_files:
    sample_path = image_files[0]
    img_pil = Image.open(sample_path).convert('RGB')
    arr_preprocessed = load_and_preprocess_image(sample_path, image_size=IMG_SIZE)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Image originale
    axes[0].imshow(img_pil, cmap='gray' if img_pil.mode == 'L' else None)
    axes[0].set_title(f'Image originale\n{img_pil.size[0]}×{img_pil.size[1]} px')
    axes[0].axis('off')

    # Après prétraitement
    axes[1].imshow(arr_preprocessed)
    axes[1].set_title(f'Prétraitée\nshape={arr_preprocessed.shape}')
    axes[1].axis('off')

    # Histogramme des pixels
    axes[2].hist(arr_preprocessed.ravel(), bins=50, color='#2980b9', alpha=0.8)
    axes[2].set_title(f'Distribution des pixels\nmin={arr_preprocessed.min():.2f}  max={arr_preprocessed.max():.2f}')
    axes[2].set_xlabel('Valeur pixel')
    axes[2].set_ylabel('Fréquence')

    plt.suptitle(f'Pipeline de prétraitement — {sample_path.name}', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Aucune image pour tester le prétraitement.")


In [ ]:
## 6. Contrôle des tailles d'images sources
if image_files:
    sample_size = min(500, len(image_files))
    sampled = np.random.choice(image_files, sample_size, replace=False)
    size_counts = Counter(Image.open(p).size for p in sampled)

    df_sizes = pd.DataFrame([
        {'width': w, 'height': h, 'count': c}
        for (w, h), c in sorted(size_counts.items(), key=lambda x: -x[1])
    ]).head(15)

    print(f"Top résolutions (sur {sample_size} images échantillonnées) :")
    print(df_sizes.to_string(index=False))

    fig, ax = plt.subplots(figsize=(8, 4))
    labels  = [f"{int(r.width)}×{int(r.height)}" for _, r in df_sizes.iterrows()]
    ax.bar(range(len(labels)), df_sizes['count'], color='#16a085')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Occurrences')
    ax.set_title(f'Distribution des tailles d\'images sources (n={sample_size})')
    plt.tight_layout()
    plt.show()


In [ ]:
## 7. Construction des datasets TensorFlow
def make_tf_dataset(split_name: str, shuffle: bool = False):
    """Crée un tf.data.Dataset depuis le répertoire d'un split."""
    split_dir = DATA_DIR / split_name
    if not split_dir.exists():
        # Essai avec majuscule (certains datasets Kaggle utilisent 'Train')
        split_dir = DATA_DIR / split_name.capitalize()
    if not split_dir.exists():
        return None
    ds = tf.keras.utils.image_dataset_from_directory(
        split_dir,
        label_mode='binary',
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH,
        shuffle=shuffle,
        seed=42,
    )
    return ds.prefetch(tf.data.AUTOTUNE)

AUTOTUNE = tf.data.AUTOTUNE

# Normalisation EfficientNet
def preprocess(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, label

train_ds = make_tf_dataset('train', shuffle=True)
val_ds   = make_tf_dataset('val')
test_ds  = make_tf_dataset('test')

if train_ds is not None:
    train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
    if val_ds  : val_ds   = val_ds.map(preprocess,  num_parallel_calls=AUTOTUNE)
    if test_ds : test_ds  = test_ds.map(preprocess,  num_parallel_calls=AUTOTUNE)
    print("Datasets créés :")
    print(f"  train : {train_ds}")
    print(f"  val   : {val_ds}")
    print(f"  test  : {test_ds}")
else:
    print("⚠ Dataset non disponible — skipping model training.")


In [ ]:
## 8. Construction du modèle baseline EfficientNetB0
def build_model(trainable_base: bool = False):
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = trainable_base

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='pneumonia_logit')(x)

    model = tf.keras.Model(inputs, outputs, name='efficientnetb0_chest_xray')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )
    return model

model = build_model(trainable_base=False)
model.summary()
print(f"\nParamètres totaux    : {model.count_params():,}")
print(f"Paramètres trainable : {sum(p.numpy().size for p in model.trainable_weights):,}")


In [ ]:
## 9. Entraînement — Phase 1 (backbone gelé, 10 époques)
PHASE1_EPOCHS = 10

if train_ds is not None:
    callbacks_p1 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=3, restore_best_weights=True, mode='max'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=2, verbose=1
        ),
    ]
    history_p1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE1_EPOCHS,
        callbacks=callbacks_p1,
        verbose=1,
    )
    print(f"\nPhase 1 terminée — val_acc={history_p1.history['val_accuracy'][-1]:.4f}"
          f"  val_auc={history_p1.history['val_auc'][-1]:.4f}")
else:
    history_p1 = None
    print("Skipped (dataset absent).")


In [ ]:
## 10. Fine-tuning — Phase 2 (dégelement des 30 dernières couches)
PHASE2_EPOCHS = 10

if train_ds is not None:
    # Dégeler uniquement les couches supérieures du backbone
    model.layers[1].trainable = True
    fine_tune_at = len(model.layers[1].layers) - 30
    for layer in model.layers[1].layers[:fine_tune_at]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )

    trainable_count = sum(p.numpy().size for p in model.trainable_weights)
    print(f"Paramètres trainable après dégelement : {trainable_count:,}")

    callbacks_p2 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=5, restore_best_weights=True, mode='max'
        ),
    ]
    history_p2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=PHASE2_EPOCHS,
        callbacks=callbacks_p2,
        initial_epoch=len(history_p1.history['loss']) if history_p1 else 0,
        verbose=1,
    )
    print(f"\nPhase 2 terminée — val_acc={history_p2.history['val_accuracy'][-1]:.4f}"
          f"  val_auc={history_p2.history['val_auc'][-1]:.4f}")
else:
    history_p2 = None
    print("Skipped (dataset absent).")


In [ ]:
## 11. Courbes d'apprentissage
def plot_history(histories, labels):
    """Trace accuracy, loss, AUC sur plusieurs phases."""
    keys = ['accuracy', 'loss', 'auc']
    fig, axes = plt.subplots(1, len(keys), figsize=(15, 4))
    offset = 0
    for hist, label in zip(histories, labels):
        if hist is None:
            continue
        n = len(hist.history['loss'])
        xs = range(offset, offset + n)
        for ax, key in zip(axes, keys):
            ax.plot(xs, hist.history[key],     label=f'{label} train', linewidth=2)
            ax.plot(xs, hist.history[f'val_{key}'], label=f'{label} val',
                    linestyle='--', linewidth=2)
        offset += n

    for ax, key in zip(axes, keys):
        ax.set_title(key.capitalize())
        ax.legend(fontsize=7)
        ax.set_xlabel('Époque')
        ax.grid(alpha=0.3)

    plt.suptitle('Courbes d\'apprentissage — Chest X-ray', fontweight='bold')
    plt.tight_layout()
    plt.show()

if history_p1 or history_p2:
    plot_history(
        [history_p1, history_p2],
        ['Phase 1 (gelé)', 'Phase 2 (fine-tune)']
    )


In [ ]:
## 12. Évaluation sur le jeu de test
if test_ds is not None:
    # Collecte labels et prédictions
    y_true, y_pred_prob = [], []
    for images, labels in test_ds:
        preds = model.predict(images, verbose=0).ravel()
        y_true.extend(labels.numpy().ravel().tolist())
        y_pred_prob.extend(preds.tolist())

    y_true      = np.array(y_true)
    y_pred_prob = np.array(y_pred_prob)
    y_pred      = (y_pred_prob > 0.5).astype(int)

    # Rapport de classification
    print("=== Rapport de classification ===")
    print(classification_report(y_true, y_pred, target_names=CLASSES))

    # ROC-AUC
    fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
    roc_auc     = auc(fpr, tpr)

    # Matrice de confusion
    cm = confusion_matrix(y_true, y_pred)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Courbe ROC
    axes[0].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {roc_auc:.4f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
    axes[0].set_xlabel('Taux faux positifs')
    axes[0].set_ylabel('Taux vrais positifs')
    axes[0].set_title('Courbe ROC — Test set')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Matrice de confusion
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
    axes[1].set_title('Matrice de confusion — Test set')

    plt.tight_layout()
    plt.show()

    print(f"\nRésumé métriques test :")
    print(f"  Accuracy  : {(y_pred == y_true).mean():.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    tn, fp, fn, tp = cm.ravel()
    print(f"  Sensitivité (Recall PNEUMONIA) : {tp / (tp + fn):.4f}")
    print(f"  Spécificité (Recall NORMAL)    : {tn / (tn + fp):.4f}")
else:
    print("Test set non disponible.")


## 13. Synthese decisionnelle

Ce notebook ne doit pas etre lu comme une promesse de performance, mais comme une trace d'experimentation avant mise en production. Tant qu'il n'a pas ete execute de bout en bout sur une version de donnees figee, aucune affirmation de performance ne doit etre reprise dans l'API, la documentation ou l'interface utilisateur.

## Conditions minimales pour poursuivre

- verifier que le split est stable et qu'il n'existe pas de fuite evidente entre train, validation et test ;
- documenter la version exacte de la configuration, du dataset et du modele exporte ;
- choisir le seuil de decision sur la validation, pas sur le test ;
- analyser manuellement les faux negatifs `PNEUMONIA`, car ce sont eux qui portent le risque principal.

## Criteres go / no-go avant integration

- `ROC-AUC`, sensibilite et specificite doivent etre rapportees ensemble ;
- la sensibilite cible doit etre validee avec le cas d'usage clinique vise ;
- les cartes d'activation du notebook Grad-CAM doivent indiquer un focus plausible sur les zones pulmonaires, et non sur des artefacts de bord ou du texte incruste ;
- un jeu de test externe ou au minimum un split patient-wise doit etre prevu avant de parler de pre-production.

## Suite logique

1. lancer une execution versionnee via le pipeline d'entrainement ;
2. comparer cette baseline a ConvNeXt et aux approches plus explicables ;
3. consigner les erreurs recurrentes pour savoir si le verrou est dans les donnees, le modele ou le seuil de decision.